# nb24 — Google Trends Quick Comparison

Does adding Google Trends search volume (l2 norm of first top-20 hit) improve prediction?
More importantly: is the model using the **trend values** or just learning that **some artists have data**?

**Three experiments:**
1. **Baseline** — same 26 features as nb21, no trends
2. **Trends (all artists)** — add `l2norm` + `has_trends` flag; NaN for missing
3. **Trends-only subset** — only artists WITH trend data; compare with vs without l2norm

Models: XGBoost and CatBoost with nb21 pipeline params (no re-tuning).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
N_SPLITS = 5

In [ ]:
# --- Load and join ---
df_artists = pd.read_csv('df_artists.csv')
df_songs   = pd.read_csv('GS/df_songs_google_decay.csv')

# For each artist: l2norm of their first top-20 hit that has trend data
# (same join logic as ml_sandbox_8 cell 2)
top20_trends = (
    df_songs[(df_songs['peak_pos'] <= 20) & (df_songs['google_trend_l2norm'].notna())]
    .sort_values('first_charting_month_and_year')
    .groupby('name', as_index=False).first()[['name', 'google_trend_l2norm']]
    .rename(columns={'google_trend_l2norm': 'l2norm'})
)
df_artists = df_artists.merge(top20_trends, on='name', how='left')

# --- Filter: first top-20 hit 2000-2019, non-null target ---
df = (
    df_artists[df_artists['first_top_20_hit_year'].between(2000, 2019)]
    .dropna(subset=['top_20_hitmaker'])
    .drop_duplicates()
    .reset_index(drop=True)
)

# --- Genre one-hot encoding (same 20 categories as df_artists_final) ---
GENRES = [
    'Blues', 'Classical', 'Country/Americana', 'Easy Listening/Vocal',
    'Electronic/Dance', 'Experimental/Avant-Garde', 'Folk',
    'Gospel/Christian/Religious', 'Hip Hop/Rap', 'Jazz', 'Latin', 'Metal',
    'Pop', 'Punk/Hardcore', 'R&B/Soul/Funk', 'Reggae/Caribbean', 'Rock', 'World Music',
]
genre_str = df['combined_major_genre_categories'].fillna('')
df['#_of_genres_artist'] = genre_str.apply(
    lambda x: len([g for g in x.split(', ') if g.strip()]))
for g in GENRES:
    df[f'artist_genre_{g}'] = genre_str.apply(
        lambda x: int(g in [s.strip() for s in x.split(',')]))
df['artist_genre_unknown'] = (
    df['combined_major_genre_categories'].isna() |
    (df['combined_major_genre_categories'].str.strip() == '')
).astype(int)

# --- Network centrality: fill nulls with 0 ---
CENTRALITY = [
    'harmonic_closeness_centrality_top20_rolling5',
    'betweenness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
]
df[CENTRALITY] = df[CENTRALITY].fillna(0)

# --- Trends flag ---
df['has_trends'] = df['l2norm'].notna().astype(int)

# --- Final feature set ---
BASELINE_FEATURES = (
    ['years_through_first_top_20_hit',
     '#_of_charting_songs_through_first_top_20_hit',
     'top_20_hit_song_#_wks_on_chart_any_position']
    + [f'artist_genre_{g}' for g in GENRES]
    + ['artist_genre_unknown', '#_of_genres_artist']
    + CENTRALITY
)
TARGET = 'top_20_hitmaker'

df_model = df[BASELINE_FEATURES + ['l2norm', 'has_trends', TARGET]].copy()
df_model[TARGET] = df_model[TARGET].astype(int)

print(f'Dataset shape: {df_model.shape}')
print(f'Artists with trends: {df_model["has_trends"].sum()} / {len(df_model)} '
      f'({df_model["has_trends"].mean():.1%})')
print(f'Class balance:\n{df_model[TARGET].value_counts(normalize=True).round(3)}')

In [ ]:
# --- Confound pre-check: is missingness correlated with hitmaker status? ---
y = df_model[TARGET]

presence = df_model.groupby('has_trends')[TARGET].agg(['mean','count'])
presence.index = ['No trends data', 'Has trends data']
presence.columns = ['Hitmaker rate', 'N artists']
print('Hitmaker rate by trends presence:')
print(presence.to_string())
print()

by_class = df_model.groupby(TARGET)['has_trends'].mean()
print(f'Trends data present in one-hit wonders: {by_class[0]:.1%}')
print(f'Trends data present in hitmakers:       {by_class[1]:.1%}')
print()

# Quick single-feature AUC test
from sklearn.model_selection import cross_val_score
mask_has = df_model['has_trends'] == 1
single_auc = cross_val_score(
    XGBClassifier(n_estimators=50, random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
    df_model[['l2norm']].fillna(-1),
    y, cv=5, scoring='roc_auc'
).mean()
flag_auc = cross_val_score(
    XGBClassifier(n_estimators=50, random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
    df_model[['has_trends']],
    y, cv=5, scoring='roc_auc'
).mean()
print(f'Single-feature AUC — l2norm alone (NaN→-1):  {single_auc:.4f}')
print(f'Single-feature AUC — has_trends flag alone:  {flag_auc:.4f}')
print()
print('If flag_auc ≈ single_auc → model is learning presence/absence, not values.')
print('If l2norm_auc >> flag_auc on trends-only subset → values carry real signal.')

# l2norm-only AUC on trends-only subset
df_sub = df_model[mask_has]
l2_only_auc = cross_val_score(
    XGBClassifier(n_estimators=50, random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
    df_sub[['l2norm']],
    df_sub[TARGET], cv=5, scoring='roc_auc'
).mean()
print(f'\nSingle-feature AUC — l2norm on trends-only subset (n={mask_has.sum()}): {l2_only_auc:.4f}')

In [ ]:
# --- Model params (nb21 pipeline, no re-tuning) ---
XGB_PARAMS = {
    'n_estimators': 239, 'learning_rate': 0.0238, 'max_depth': 5,
    'min_child_weight': 12, 'gamma': 4.5, 'subsample': 0.62,
    'colsample_bytree': 0.36, 'reg_alpha': 0.36, 'reg_lambda': 0.0,
    'random_state': RANDOM_STATE, 'eval_metric': 'logloss', 'verbosity': 0,
}
CAT_PARAMS = {
    'n_estimators': 50, 'learning_rate': 0.06180643097470682,
    'depth': 3, 'l2_leaf_reg': 4.970472180919757,
    'random_strength': 3.657777929321733, 'min_data_in_leaf': 9,
    'border_count': 178, 'random_state': RANDOM_STATE, 'verbose': 0,
}

def run_cv(X, y, model_name, n_splits=N_SPLITS):
    """Stratified k-fold CV. Returns dict with train/val AUC, std, gap."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    tr_aucs, val_aucs = [], []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        imp = SimpleImputer(strategy='median')
        X_tr_i  = pd.DataFrame(imp.fit_transform(X_tr),  columns=X_tr.columns)
        X_val_i = pd.DataFrame(imp.transform(X_val),     columns=X_val.columns)
        m = XGBClassifier(**XGB_PARAMS) if model_name == 'xgb' else CatBoostClassifier(**CAT_PARAMS)
        m.fit(X_tr_i, y_tr)
        tr_aucs.append(roc_auc_score(y_tr,  m.predict_proba(X_tr_i)[:, 1]))
        val_aucs.append(roc_auc_score(y_val, m.predict_proba(X_val_i)[:, 1]))
    return {'train': np.mean(tr_aucs), 'val': np.mean(val_aucs),
            'std': np.std(val_aucs),   'gap': np.mean(tr_aucs) - np.mean(val_aucs)}

print('CV runner ready. Params loaded from nb21 pipeline.')
print(f'XGBoost: {XGB_PARAMS}')
print(f'CatBoost: {CAT_PARAMS}')

In [ ]:
# ================================================================
# Experiment 1: Baseline — no trends, all artists
# ================================================================
print('=== Experiment 1: Baseline (no trends, all artists) ===')
X1 = df_model[BASELINE_FEATURES]
y1 = df_model[TARGET]
r1x = run_cv(X1, y1, 'xgb')
r1c = run_cv(X1, y1, 'cat')
print(f'XGBoost:  Val AUC={r1x["val"]:.4f} ± {r1x["std"]:.4f}  '
      f'Train={r1x["train"]:.4f}  Gap={r1x["gap"]:+.4f}')
print(f'CatBoost: Val AUC={r1c["val"]:.4f} ± {r1c["std"]:.4f}  '
      f'Train={r1c["train"]:.4f}  Gap={r1c["gap"]:+.4f}')
print(f'(Reference from nb21: XGBoost CV AUC=0.756, CatBoost pipeline CV AUC=0.737)')

# ================================================================
# Experiment 2: All artists + l2norm + has_trends flag
# ================================================================
print('\n=== Experiment 2: All artists + l2norm + has_trends flag ===')
X2 = df_model[BASELINE_FEATURES + ['l2norm', 'has_trends']]
y2 = df_model[TARGET]
r2x = run_cv(X2, y2, 'xgb')
r2c = run_cv(X2, y2, 'cat')
print(f'XGBoost:  Val AUC={r2x["val"]:.4f} ± {r2x["std"]:.4f}  '
      f'Train={r2x["train"]:.4f}  Gap={r2x["gap"]:+.4f}  '
      f'Δvs baseline={r2x["val"]-r1x["val"]:+.4f}')
print(f'CatBoost: Val AUC={r2c["val"]:.4f} ± {r2c["std"]:.4f}  '
      f'Train={r2c["train"]:.4f}  Gap={r2c["gap"]:+.4f}  '
      f'Δvs baseline={r2c["val"]-r1c["val"]:+.4f}')

# ================================================================
# Experiment 3: Only artists WITH trends data — with vs without l2norm
# ================================================================
mask_trends = df_model['has_trends'] == 1
df_sub = df_model[mask_trends].reset_index(drop=True)
y_sub  = df_sub[TARGET]
print(f'\n=== Experiment 3: Trends-only subset (n={mask_trends.sum()}) ===')
print(f'Class balance in subset: {y_sub.value_counts(normalize=True).round(3).to_dict()}')

r3bx = run_cv(df_sub[BASELINE_FEATURES],              y_sub, 'xgb')  # no l2norm
r3x  = run_cv(df_sub[BASELINE_FEATURES + ['l2norm']], y_sub, 'xgb')  # with l2norm
r3bc = run_cv(df_sub[BASELINE_FEATURES],              y_sub, 'cat')
r3c  = run_cv(df_sub[BASELINE_FEATURES + ['l2norm']], y_sub, 'cat')

print(f'XGBoost  w/o l2norm: {r3bx["val"]:.4f}  →  with l2norm: {r3x["val"]:.4f}  '
      f'Δ={r3x["val"]-r3bx["val"]:+.4f}')
print(f'CatBoost w/o l2norm: {r3bc["val"]:.4f}  →  with l2norm: {r3c["val"]:.4f}  '
      f'Δ={r3c["val"]-r3bc["val"]:+.4f}')

In [ ]:
# ================================================================
# Summary table
# ================================================================
results = pd.DataFrame([
    {'Experiment': 'Exp1 Baseline (all)',          'Model': 'XGBoost',  'N': len(X1),
     'Val AUC': r1x['val'], 'Std': r1x['std'], 'Gap': r1x['gap']},
    {'Experiment': 'Exp1 Baseline (all)',          'Model': 'CatBoost', 'N': len(X1),
     'Val AUC': r1c['val'], 'Std': r1c['std'], 'Gap': r1c['gap']},
    {'Experiment': 'Exp2 + l2norm + flag (all)',   'Model': 'XGBoost',  'N': len(X2),
     'Val AUC': r2x['val'], 'Std': r2x['std'], 'Gap': r2x['gap']},
    {'Experiment': 'Exp2 + l2norm + flag (all)',   'Model': 'CatBoost', 'N': len(X2),
     'Val AUC': r2c['val'], 'Std': r2c['std'], 'Gap': r2c['gap']},
    {'Experiment': 'Exp3 Baseline (trends-only)',  'Model': 'XGBoost',  'N': mask_trends.sum(),
     'Val AUC': r3bx['val'],'Std': r3bx['std'],'Gap': r3bx['gap']},
    {'Experiment': 'Exp3 Baseline (trends-only)',  'Model': 'CatBoost', 'N': mask_trends.sum(),
     'Val AUC': r3bc['val'],'Std': r3bc['std'],'Gap': r3bc['gap']},
    {'Experiment': 'Exp3 + l2norm (trends-only)',  'Model': 'XGBoost',  'N': mask_trends.sum(),
     'Val AUC': r3x['val'], 'Std': r3x['std'], 'Gap': r3x['gap']},
    {'Experiment': 'Exp3 + l2norm (trends-only)',  'Model': 'CatBoost', 'N': mask_trends.sum(),
     'Val AUC': r3c['val'], 'Std': r3c['std'], 'Gap': r3c['gap']},
])
results[['Val AUC','Std','Gap']] = results[['Val AUC','Std','Gap']].round(4)
print(results.to_string(index=False))

In [ ]:
# ================================================================
# Confound diagnosis: does has_trends dominate l2norm in feature importance?
# ================================================================
imp = SimpleImputer(strategy='median')
X2_imp = pd.DataFrame(imp.fit_transform(X2), columns=X2.columns)

xgb_diag = XGBClassifier(**XGB_PARAMS)
xgb_diag.fit(X2_imp, y2)

imp_scores = pd.Series(xgb_diag.get_booster().get_score(importance_type='gain'))

CLEAN = {
    '#_of_charting_songs_through_first_top_20_hit': 'Charting songs',
    'top_20_hit_song_#_wks_on_chart_any_position':  'Weeks on chart',
    'years_through_first_top_20_hit':               'Years to first hit',
    '#_of_genres_artist':                           'Num genres',
    'harmonic_closeness_centrality_top20_rolling5': 'Harmonic closeness',
    'betweenness_centrality_top20_rolling5':        'Betweenness',
    'eigenvector_centrality_top20_rolling5':        'Eigenvector',
    'l2norm':     'Google Trends l2norm  ← NEW',
    'has_trends': 'Has trends data (flag)  ← NEW',
}
imp_scores.index = [CLEAN.get(i, i.replace('artist_genre_', '')) for i in imp_scores.index]
imp_top = imp_scores.sort_values(ascending=True).tail(15)

colors = ['#d73027' if 'NEW' in i else '#2166ac' for i in imp_top.index]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_top.index, imp_top.values, color=colors)
ax.set_title('Feature importance — XGBoost Exp 2 (red = Google Trends features)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Gain importance')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

l2_imp  = imp_scores.get('Google Trends l2norm  ← NEW', 0)
has_imp = imp_scores.get('Has trends data (flag)  ← NEW', 0)
print(f'Google Trends l2norm gain:     {l2_imp:.1f}')
print(f'has_trends flag gain:           {has_imp:.1f}')
print()
if has_imp > 0 and l2_imp > 0:
    ratio = has_imp / l2_imp
    if ratio > 2:
        print(f'⚠  CONFOUND: has_trends gain is {ratio:.1f}x l2norm gain.')
        print('   Model is primarily learning presence/absence, not trend magnitude.')
    elif ratio < 0.5:
        print(f'✓  l2norm gain is {1/ratio:.1f}x has_trends gain.')
        print('   Model is using the actual trend values, not just presence.')
    else:
        print(f'Mixed: has_trends/l2norm ratio = {ratio:.2f} (close to 1 — ambiguous).')
elif l2_imp == 0 and has_imp == 0:
    print('Neither trends feature was used by XGBoost at all — model ignores Google Trends entirely.')
elif l2_imp == 0:
    print('⚠  l2norm was never split on — model only uses the presence flag.')
elif has_imp == 0:
    print('✓  has_trends was never split on — model only uses actual trend values.')

In [ ]:
# ================================================================
# SHAP: direction and magnitude of l2norm and has_trends effects
# ================================================================
explainer = shap.TreeExplainer(xgb_diag)
shap_vals  = explainer.shap_values(X2_imp)
shap_df    = pd.DataFrame(shap_vals, columns=X2_imp.columns)

# Mean signed SHAP for new features
l2_shap  = shap_df['l2norm'].mean()
has_shap = shap_df['has_trends'].mean()
print(f'Mean SHAP  l2norm:     {l2_shap:+.4f}  (positive = higher l2 → more likely hitmaker)')
print(f'Mean SHAP  has_trends: {has_shap:+.4f}  (positive = having data → more likely hitmaker)')
print()
print('If has_trends SHAP > 0 and large: model rewards artists for merely having data.')
print('This is the confound — larger artists are more likely to have trend data AND to be hitmakers.')

# Quick visualization — SHAP summary for new features only
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, feat, label in [
    (axes[0], 'l2norm',     'Google Trends l2norm'),
    (axes[1], 'has_trends', 'Has trends data (flag)'),
]:
    sc = ax.scatter(X2_imp[feat], shap_df[feat], c=y2, cmap='RdBu_r',
                    alpha=0.4, s=20, vmin=0, vmax=1)
    ax.axhline(0, color='gray', lw=1, ls='--')
    ax.set_xlabel(label)
    ax.set_ylabel('SHAP value')
    ax.set_title(f'SHAP vs {label}', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
plt.colorbar(sc, ax=axes[1], label='True label (1=Hitmaker)')
fig.suptitle('SHAP dependence plots — Google Trends features\n'
             '(blue=one-hit wonder, red=hitmaker)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()